# ప్రాంప్ట్ ఇంజనీరింగ్ పరిచయం
ప్రాంప్ట్ ఇంజనీరింగ్ అనేది సహజ భాషా ప్రాసెసింగ్ పనుల కోసం ప్రాంప్ట్‌లను డిజైన్ చేయడం మరియు ఆప్టిమైజ్ చేయడం ప్రక్రియ. ఇది సరైన ప్రాంప్ట్‌లను ఎంచుకోవడం, వాటి పారామితులను సర్దుబాటు చేయడం, మరియు వాటి పనితీరును మూల్యాంకనం చేయడం కలిగి ఉంటుంది. NLP మోడళ్లలో ఉన్నత స్థాయి ఖచ్చితత్వం మరియు సామర్థ్యం సాధించడానికి ప్రాంప్ట్ ఇంజనీరింగ్ అత్యంత ముఖ్యమైనది. ఈ విభాగంలో, మేము OpenAI మోడళ్లను ఉపయోగించి ప్రాంప్ట్ ఇంజనీరింగ్ మౌలిక విషయాలను పరిశీలించబోతున్నాము.


### వ్యాయామం 1: టోకనైజేషన్  
OpenAI నుండి ఓపెన్ సోర్స్ వేగవంతమైన టోకనైజర్ అయిన tiktoken ఉపయోగించి టోకనైజేషన్‌ను పరిశీలించండి  
మరిన్ని ఉదాహరణలకు [OpenAI కుకీబుక్](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) ను చూడండి.  


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### వ్యాయామం 2: Microsoft Foundry Models కీ సెట్టింగ్‌ను సరిచూసుకోండి

> **గమనిక:** GitHub Models జూలై 2026 చివరికి రిటైర్ అవుతోంది. ఈ వ్యాయామం [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) ను ఉపయోగిస్తుంది, ఇది అదే ఉచితంగా ప్రయత్నించదగిన మోడల్ క్యాటలాగ్ మరియు Azure AI Inference SDK అనుభవాన్ని అందిస్తుంది.

మీ Microsoft Foundry Models ఎండ్‌పాయింట్ సరైనదిగా సెట్ అవడ్డాన్నిఅని నిర్ధారించడానికి క్రింది కోడ్ నడిపించండి. కోడ్ ఒక సాదారణ ప్రాంప్ట్ ప్రయత్నించి పూర్తయినదో లేదో చెక్ చేస్తుంది. ఇన్‌పుట్ `oh say can you see` కి సమాధానంగా `by the dawn's early light..` వంటి పూర్తి వాక్యం వచ్చేలా ఉంటుంది.


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

model_name = "gpt-4o-mini"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### వ్యాయామం 3: సృష్టికర్తలు
మీరు LLM ను ప్రాంప్ట్ కోసం పూర్తి వివరాలను ఇవ్వమని అడిగినప్పుడు ఏం జరుగుతుందో అన్వేషించండి, అది ఉండకపోవచ్చు లేదా అది ముందుగా శిక్షణ పొందిన డేటాసెట్ (ఇంకా తాజా) లో లేని కారణంగా తెలియకపోవచ్చు. మీరు విభిన్నమైన ప్రాంప్ట్ లేదా విభిన్న మోడల్ ను ప్రయత్నించినపుడు స్పందన ఎలా మారుతుందో చూడండి.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### వ్యాయామం 4: ఆదేశాల ప్రాతిపదికగా 
ప్రాథమిక విషయాన్ని సెట్ చేయడానికి "text" వేరియబుల్ ను ఉపయోగించండి 
మరియు ఆ ప్రాథమిక విషయానికి సంబంధించి ఆదేశం ఇవ్వడానికి "prompt" వేరియబుల్ ను ఉపయోగించండి.

ఇక్కడ మేము మోడల్‌ని రెండవ తరగతి విద్యార్థికి టెక్ట్స్ ను సారాంశం చేయమని అడుగుతున్నాము


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### వ్యాయామం 5: సంక్లిష్ట ప్రాంప్ట్  
సిస్టమ్, యూజర్ మరియు అసిస్టెంట్ సందేశాలు ఉన్న అభ్యర్థనను ప్రయత్నించండి  
అసిస్టెంట్ సందర్భాన్ని సిస్టమ్ సెట్చేస్తుంది  
యూజర్ & అసిస్టెంట్ సందేశాలు బహుళ తరం సంభాషణ సందర్భాన్ని అందిస్తాయి  

సిస్టమ్ సందర్భంలో అసిస్టెంట్ వ్యక్తిత్వం "వేదనాత్మకంగా" ఎలా సెట్ చేయబడిందో గమనించండి.  
వేరే వ్యక్తిత్వ సందర్భాన్ని ప్రయత్నించండి. లేదా వేరే సిరీస్ ఇన్‌పుట్/ఔట్‌పుట్ సందేశాలను ప్రయత్నించండి  


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### వ్యాయామం: మీ సహజజ్ఞానాన్ని అన్వేషించండి
పైన ఉన్న ఉదాహరణలు మీరు కొత్త ప్రాంప్ట్‌లను (సాధారణ, సంక్లిష్ట, సూచన మొదలయినవి) తయారు చేయడానికి ఉపయోగించుకోవచ్చని నమూనాలను ఇవ్వగలవు - మేము చర్చించిన ఇతర ఆలోచనలన్నింటిని కూడా అన్వేషించడానికి ఇతర వ్యాయామాలను తయారు చేయండి, ఉదాహరణలు, సంకేతాలు మరియు మరెన్నో.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
